# 01 — Data cleaning & preparation (version R1, corrigée)

Copie de `Pipeline_analyse/01_Data_Cleaning_Preparation.ipynb` (conservé intact) avec les corrections de la révision R1. Chaque correction est signalée dans le code par `# R1-FIX`.

| # | Cellule | Correction |
|---|---|---|
| 1 | §3 | Biologie restreinte au séjour en réanimation (de la veille de l'admission au jour de sortie). Avant : toutes les valeurs du patient dans l'extraction 2012-2023, y compris d'autres séjours. Paramètre `RESTRICT_TO_STAY`. |
| 2 | §4 / §7 | Les natrémies ne sont plus perdues quand la créatinine du même prélèvement est non numérique. |
| 3 | §5 | CKD-EPI 2009 corrigée (`min`/`max`, facteur féminin 1,018) et rétro-calcul MDRD (DFG 75) corrigé. |
| 4 | §9 | Première date d'hypo/hypernatrémie : tri chronologique avant de prendre la première valeur. |
| 5 | §14 | Import sklearn inutilisé supprimé. |
| 6 | §2 | « Severe burn » recalculé comme TBSA ≥ 20 % (définition des Méthodes) au lieu de la variable saisie dans le CRF. |

Sorties : `outputs/` de ce dossier (`ACCPM/R1/outputs`).

## 0. Librairies

In [1]:

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)   # avertissements pandas (groupby.apply) sans effet sur les resultats

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

DATA = "../../Pipeline_analyse/data/raw"
RESTRICT_TO_STAY = True   # R1-FIX 1 : biologie limitee au sejour en reanimation
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)


## 1. Import des fichiers bruts

Creatinine/sodium/bio sanguine, glycemie, biologie urinaire (2012-2023),
traitements diuretiques et apports hydriques.


In [2]:

years = ['2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022']

##Import files bio all
frames_bio = [pd.read_excel(f"{DATA}/CTB_Ext_2012-2023/Raw/{y}_SLS_CTB.xlsx") for y in years]
frames_bio += [
    pd.read_excel(f"{DATA}/CTB_Ext_2012-2023/Raw/2023_SLS_CTB-V8.xlsx"),
    pd.read_excel(f"{DATA}/CTB_Ext_2012-2023/Raw/2023_SLS_CTB-V9.xlsx"),
]
dfglimps = pd.concat(frames_bio, ignore_index=True)

#import gluc_bio
frames_gluc = [pd.read_excel(f"{DATA}/CTB_Ext_2012-2023/Glu/{y}_SLS_gluc.xlsx") for y in years]
frames_gluc += [
    pd.read_excel(f"{DATA}/CTB_Ext_2012-2023/Glu/2023_SLS_glucV8.xlsx"),
    pd.read_excel(f"{DATA}/CTB_Ext_2012-2023/Glu/2023_SLS_glucV9.xlsx"),
]
dfglimpsgluc = pd.concat(frames_gluc, ignore_index=True)

#import urine_bio
frames_urine = [pd.read_excel(f"{DATA}/CTB_Ext_2012-2023/DonneeU/{y}_SLS_CTB_URINES.xlsx") for y in years]
frames_urine += [
    pd.read_excel(f"{DATA}/CTB_Ext_2012-2023/DonneeU/2023_SLS_CTB-V8_URINES.xlsx"),
    pd.read_excel(f"{DATA}/CTB_Ext_2012-2023/DonneeU/2023_SLS_CTB-V9_URINES.xlsx"),
]
dfglimpsurine = pd.concat(frames_urine, ignore_index=True)

urine_cols = [
    'VU24  :  Volume urines des 24h',
    'NAU  :  Sodium',
    'NAU24  :  Sodium /24h',
    'KU  :  Potassium',
    'KU24  :  Potassium /24h',
    'CREU  :  Créatinine',
    'URU  :  Urée',
    'PROTU  :  Protéines',
    'NGAL_U  :  NGAL U'
]

dfglimpsurine[urine_cols] = (
    dfglimpsurine[urine_cols]
    .astype(str)
    .replace(r'^\s*<\s*', '', regex=True)
    .apply(pd.to_numeric, errors='coerce')
)

#import diuretic treatment
df_loop = pd.read_excel(f'{DATA}/Mdt/daily_loop.xlsx')

#import imputs treatments
df_inputs = pd.read_excel(f'{DATA}/Mdt/daily_inputs.xlsx')

print("dfglimps (bio sanguine)   :", dfglimps.shape)
print("dfglimpsgluc (glycemie)   :", dfglimpsgluc.shape)
print("dfglimpsurine (urines)    :", dfglimpsurine.shape)
print("df_loop (diuretiques)     :", df_loop.shape)
print("df_inputs (apports)       :", df_inputs.shape)


dfglimps (bio sanguine)   : (127193, 46)
dfglimpsgluc (glycemie)   : (16522, 64)
dfglimpsurine (urines)    : (23337, 17)
df_loop (diuretiques)     : (4341, 13)
df_inputs (apports)       : (17511, 13)


## 2. Base patient (CRF) et selection de la cohorte

`BDD_Final_LB_tomodify.xlsx` contient toutes les variables demographiques,
comorbidites et caracteristiques de brulure deja saisies manuellement
(Age, Sexe, TBSA, Charlson, IGSII, ABSI, RRT, deces, etc.). On restreint aux
patients presents dans les extraits labo (creatinine/sodium).


In [3]:

dfZoe = pd.read_excel(f"{DATA}/BDD_Final_LB_tomodify.xlsx")
list_IPP_glimps = dfglimps['NIP'].unique().tolist()
df = dfZoe[dfZoe['IPP2'].isin(list_IPP_glimps)].copy()
print("Patients dans la base CRF           :", len(dfZoe))
print("Patients avec donnees labo (cohorte):", len(df))


# R1-FIX 6 : "severe burn" recalcule selon la definition des Methodes (TBSA >= 20 %).
# La variable saisie dans le CRF ne suivait pas cette definition (TBSA = 20 % : 12 oui / 60 non ; 10 "severes" avec TBSA < 20 %).
print("severe.burn (CRF) :", int(df['severe.burn'].sum()), "-> TBSA >= 20 % :", int((df['TBSA'] >= 20).sum()))
df['severe.burn'] = (df['TBSA'] >= 20).astype(int)

Patients dans la base CRF           : 1764
Patients avec donnees labo (cohorte): 1764
severe.burn (CRF) : 468 -> TBSA >= 20 % : 518


## 3. Format des dates

In [4]:

df['D.entry'] = pd.to_datetime(df['D.entry'], format='%d-%m-%y %Hh%Mmin%Ss')
df['D.discharge'] = pd.to_datetime(df['D.discharge'], format='%d-%m-%y %Hh%Mmin%Ss')
dfglimps = pd.merge(dfglimps, df[['IPP2', 'D.entry', 'D.discharge']], how="inner", left_on=["NIP"], right_on=["IPP2"])
dfglimpsgluc = pd.merge(dfglimpsgluc, df[['IPP2', 'D.entry', 'D.discharge']], how="inner", left_on=["NIP"], right_on=["IPP2"])
dfglimps['Date -heure de prvt-'] = pd.to_datetime(dfglimps['Date -heure de prvt-'], format='%d-%m-%y %Hh%Mmin%Ss')
df['D.RRT'] = pd.to_datetime(df['D.RRT'], format='%d-%m-%y %Hh%Mmin%Ss', errors='coerce')
df['D.death'] = pd.to_datetime(df['D.death'], format='%d-%m-%y %Hh%Mmin%Ss', errors='coerce')


# R1-FIX 1 : restriction de la biologie au sejour (veille de l'admission -> jour de sortie)
dfglimpsgluc['Date de Prélèvement'] = pd.to_datetime(dfglimpsgluc['Date de Prélèvement'], errors='coerce')
def _in_stay(d, date_col):
    day = d[date_col].dt.normalize()
    return (day >= d['D.entry'].dt.normalize() - pd.Timedelta(days=1)) & (day <= d['D.discharge'].dt.normalize())
_n0, _g0 = len(dfglimps), len(dfglimpsgluc)
if RESTRICT_TO_STAY:
    dfglimps = dfglimps[_in_stay(dfglimps, 'Date -heure de prvt-')].copy()
    dfglimpsgluc = dfglimpsgluc[_in_stay(dfglimpsgluc, 'Date de Prélèvement')].copy()
print(f"Biologie sanguine conservee : {len(dfglimps)}/{_n0} ({100*len(dfglimps)/_n0:.1f} %) ; gaz du sang : {len(dfglimpsgluc)}/{_g0} ({100*len(dfglimpsgluc)/_g0:.1f} %)")

Biologie sanguine conservee : 101653/104790 (97.0 %) ; gaz du sang : 13001/13387 (97.1 %)


## 4. Selection des creatinines (entree, derniere, J28, J90)

In [5]:

def is_numeric(value):
    try:
        pd.to_numeric(value)
        return True
    except (TypeError, ValueError):
        return False
dfglimps_all = dfglimps.copy()   # R1-FIX 2 : toutes les lignes (pour la natremie)
dfglimps_flt = dfglimps[dfglimps['CREAT  :  Créatinine'].apply(is_numeric)]

### Creat entry
def select_first_creatinine_admission(group):
    sorted_creatinine = group.sort_values(by='Date -heure de prvt-', ascending=True)
    if not sorted_creatinine.empty:
        first_creatinine_value = sorted_creatinine.iloc[0]['CREAT  :  Créatinine']
        return pd.Series([first_creatinine_value], index=['Creat.entry'])
    else:
        return pd.Series([None], index=['Creat.entry'])

Creat_entry = dfglimps_flt.groupby('IPP2').apply(select_first_creatinine_admission).reset_index()

### Creat dis/last
def select_last_creatinine(group):
    sorted_creatinine = group.sort_values(by='Date -heure de prvt-', ascending=False)
    if not sorted_creatinine.empty:
        last_creatinine_value = sorted_creatinine.iloc[0]['CREAT  :  Créatinine']
        last_creatinine_date = sorted_creatinine.iloc[0]['Date -heure de prvt-']
        return pd.Series([last_creatinine_value, last_creatinine_date], index=['creat_last', 'date_last'])
    else:
        return pd.Series([None, None], index=['creat_last', 'date_last'])

result_last_creatinine = dfglimps_flt.groupby('IPP2').apply(select_last_creatinine).reset_index()

def select_creatinine_28(group):
    mask = (group['Date -heure de prvt-'] - group['D.entry']).dt.days <= 28
    filtered_group = group[mask].sort_values(by='Date -heure de prvt-', ascending=False)
    if not filtered_group.empty:
        creat_last_28 = filtered_group['CREAT  :  Créatinine'].iloc[0]
        date_creat_28 = filtered_group['Date -heure de prvt-'].iloc[0]
        return pd.Series([creat_last_28, date_creat_28], index=['creat_28', 'date_28'])
    else:
        return pd.Series([None, None], index=['creat_28', 'date_28'])

result_28 = dfglimps_flt.groupby('IPP2').apply(select_creatinine_28).reset_index()

def select_creatinine_90(group):
    mask = (group['Date -heure de prvt-'] - group['D.entry']).dt.days <= 90
    filtered_group = group[mask].sort_values(by='Date -heure de prvt-', ascending=False)
    if not filtered_group.empty:
        creat_last_90 = filtered_group['CREAT  :  Créatinine'].iloc[0]
        date_creat_90 = filtered_group['Date -heure de prvt-'].iloc[0]
        return pd.Series([creat_last_90, date_creat_90], index=['creat_90', 'date_90'])
    else:
        return pd.Series([None, None], index=['creat_90', 'date_90'])

result_90 = dfglimps_flt.groupby('IPP2').apply(select_creatinine_90).reset_index()

df = pd.merge(df, result_last_creatinine, how='left', on='IPP2')
df = pd.merge(df, result_28, how='left', on='IPP2')
df = pd.merge(df, result_90, how='left', on='IPP2')
df = pd.merge(df, Creat_entry, how='left', on='IPP2')


## 5. eGFR, creatinine de base, MAKE28/MAKE90

**Bug 1 corrige ici** : `creat_baseline()` retourne desormais bien
`sCr_baseline` (et non plus `eGFR_adm`).


In [6]:

###MAKE
### Definition egfr
def calculate_eGFR(row):
    Sex = row['Sex']
    Age = row['Age']
    sCr = row['Creat.entry']
    try:
        sCr = float(sCr) / 88.4
    except ValueError:
        return np.nan

    if Sex == 0:  # Female
        alpha = -0.329
        kappa = 0.7
    else:  # Male
        alpha = -0.411
        kappa = 0.9

    # R1-FIX 3 : CKD-EPI 2009 (min puis max ; facteur feminin 1.018)
    eGFR = 141 * (min(sCr/kappa, 1) ** alpha) * (max(sCr/kappa, 1) ** -1.209) * (0.993 ** Age) * (1.018 if Sex == 0 else 1.0)

    return eGFR

df['eGFR_adm'] = df.apply(calculate_eGFR, axis=1).astype(float)

def back_calculate_MDRD(row):
    Sex = row['Sex']
    Age = row['Age']
    eGFR = row['eGFR_adm']

    if Sex == 0:
        female_factor = 0.742
    else:
        female_factor = 1.0

    # R1-FIX 3 : inversion de MDRD pour un DFG de reference de 75 mL/min/1.73 m2 (resultat en umol/L)
    sCr = 88.4 * (75 / (175 * Age ** (-0.203) * female_factor)) ** (-1 / 1.154)

    return sCr

df['back_calculate'] = df.apply(back_calculate_MDRD, axis=1).astype(float)

def creat_baseline(row):
    eGFRadm = row['eGFR_adm']
    sCradm = row['Creat.entry']
    sCrback = row['back_calculate']
    CKD = row['renal.disease']

    if (eGFRadm < 75) & (CKD != 1):
        sCr_baseline = sCrback
    else:
        sCr_baseline = sCradm
    return sCr_baseline  # <-- CORRIGE (etait "return eGFRadm")

df['creat_baseline'] = df.apply(creat_baseline, axis=1).astype(float)

def MAKE28(row):
    RRT = row['RRT']
    D_RRT = row['D.RRT']
    D_entry = row['D.entry']
    creat_28 = row['creat_28']
    creat_baseline = row['creat_baseline']
    death = row['death']
    D_death = row['D.death']

    RRT28_D = (D_RRT - D_entry).days if pd.notna(D_RRT) and pd.notna(D_entry) else None

    try:
        sCr_D = float(creat_28) / float(creat_baseline) if pd.notna(creat_28) and pd.notna(creat_baseline) else None
    except ValueError:
        sCr_D = None

    death_D = (D_death - D_entry).days if pd.notna(D_death) and pd.notna(D_entry) else None

    RRT28 = (RRT == 1) and (RRT28_D is not None) and (RRT28_D <= 28)
    death28 = (death == 1) and (death_D is not None) and (death_D <= 28)
    NR28 = (sCr_D is not None) and (sCr_D >= 1.5)

    return (RRT28 or death28 or NR28)


df['creat_28'] = pd.to_numeric(df['creat_28'], errors='coerce')
df['creat_baseline'] = pd.to_numeric(df['creat_baseline'], errors='coerce')
df['MAKE28'] = df.apply(MAKE28, axis=1)

def MAKE90(row):
    RRT = row['RRT']
    D_RRT = row['D.RRT']
    D_entry = row['D.entry']
    creat_90 = row['creat_90']
    creat_baseline = row['creat_baseline']
    death = row['death']
    D_death = row['D.death']

    RRT90_D = (D_RRT - D_entry).days if pd.notna(D_RRT) and pd.notna(D_entry) else None

    try:
        sCr_D = float(creat_90) / float(creat_baseline) if pd.notna(creat_90) and pd.notna(creat_baseline) else None
    except ValueError:
        sCr_D = None

    death_D = (D_death - D_entry).days if pd.notna(D_death) and pd.notna(D_entry) else None

    RRT90 = (RRT == 1) and (RRT90_D is not None) and (RRT90_D <= 90)
    death90 = (death == 1) and (death_D is not None) and (death_D <= 90)
    NR90 = (sCr_D is not None) and (sCr_D >= 1.5)

    return (RRT90 or death90 or NR90)


df['creat_90'] = pd.to_numeric(df['creat_90'], errors='coerce')
df['MAKE90'] = df.apply(MAKE90, axis=1)

print("MAKE28 :", int(df['MAKE28'].sum()), "/", len(df))
print("MAKE90 :", int(df['MAKE90'].sum()), "/", len(df))

_bc = (df['eGFR_adm'] < 75) & (df['renal.disease'] != 1)
print(f"Creatinine de base retro-calculee : {int(_bc.sum())}/{int(df['Creat.entry'].notna().sum())} ({100*_bc.sum()/df['Creat.entry'].notna().sum():.1f} %)")

MAKE28 : 200 / 1764
MAKE90 : 207 / 1764
Creatinine de base retro-calculee : 299/1742 (17.2 %)


## 6. Stade KDIGO de l'insuffisance renale aigue (AKI_KDIGO / AKI_stage)

**Bug 2 corrige ici** : la baseline (`creat_baseline`, en µmol/L) est
convertie en mg/dL avant d'etre comparee a `creat_mgdl` dans le calcul du
ratio KDIGO (elle ne l'etait pas dans la version originale, ce qui ecrasait
le ratio d'un facteur ~88 et empechait le stade 2 d'apparaitre).


In [7]:

dfglimps_flt = dfglimps_flt.copy()
dfglimps_flt['creat_mgdl'] = (
    pd.to_numeric(dfglimps_flt['CREAT  :  Créatinine'], errors='coerce') / 88.4
)

df_creat = dfglimps_flt.merge(
    df[['IPP2', 'creat_baseline', 'RRT', 'D.RRT']],
    on='IPP2',
    how='left'
)

df_creat['date'] = pd.to_datetime(df_creat['Date -heure de prvt-'])

def detect_kdigo_aki(group):
    group = group.sort_values('date').copy()

    baseline_umol = group['creat_baseline'].iloc[0]
    baseline = baseline_umol / 88.4 if pd.notna(baseline_umol) else np.nan  # <-- CORRIGE : conversion µmol/L -> mg/dL
    d_entry = group['D.entry'].iloc[0]
    rrt = group['RRT'].iloc[0]
    d_rrt = group['D.RRT'].iloc[0]

    aki_date = None
    aki_stage = 0

    for i, row in group.iterrows():
        sCr = row['creat_mgdl']
        date = row['date']

        if pd.isna(sCr) or pd.isna(baseline) or baseline <= 0:
            continue

        ratio = sCr / baseline

        # --- KDIGO creatinine staging ---
        if ratio >= 3 or sCr >= 4.0:
            stage = 3
        elif ratio >= 2:
            stage = 2
        elif ratio >= 1.5:
            stage = 1
        else:
            stage = 0

        # --- 0.3 mg/dL within 48h ---
        prev = group[group['date'] <= date].tail(2)
        if len(prev) == 2:
            delta = prev.iloc[1]['creat_mgdl'] - prev.iloc[0]['creat_mgdl']
            dt = (prev.iloc[1]['date'] - prev.iloc[0]['date']).total_seconds() / 3600
            if delta >= 0.3 and dt <= 48:
                stage = max(stage, 1)

        if stage > 0 and aki_date is None:
            aki_date = date
            aki_stage = stage
        elif stage > aki_stage:
            aki_stage = stage

    # --- RRT overrides ---
    if rrt == 1 and pd.notna(d_rrt):
        if aki_date is None or d_rrt < aki_date:
            aki_date = d_rrt
        aki_stage = 3

    return pd.Series({
        'AKI_KDIGO': int(aki_stage > 0),
        'AKI_stage': aki_stage,
        'AKI_date': aki_date
    })

aki_results = (
    df_creat
    .groupby('IPP2')
    .apply(detect_kdigo_aki, include_groups=False)
    .reset_index()
)

df = df.merge(aki_results, on='IPP2', how='left')

print("AKI_KDIGO :", int(df['AKI_KDIGO'].sum()), "/", len(df), f"({100*df['AKI_KDIGO'].mean():.1f}%)")
print("AKI_stage distribution :")
print(df['AKI_stage'].value_counts(dropna=False).sort_index())


AKI_KDIGO : 406 / 1764 (23.3%)
AKI_stage distribution :
AKI_stage
0.0    1336
1.0     180
2.0      74
3.0     152
NaN      22
Name: count, dtype: int64


## 7. Exclusion (hyperglycemie majeure / hypertriglyceridemie) puis correction osmotique de la natremie

Methodologie (cf. section Methodes du manuscrit) : les valeurs de natremie sont **exclues**
(non utilisees, pas seulement corrigees) si une hyperglycemie majeure (glycemie >= 10 mmol/L)
ou une hypertriglyceridemie (TG >= 2 g/L) est documentee le meme jour -- la formule de
correction osmotique n'est pas consideree fiable a ces niveaux. Les valeurs restantes recoivent
ensuite la correction osmotique standard (formule de Katz) en fonction de la glycemie du jour.


In [8]:

# ---------------------------
# 1) Build df_na_final with 3 sodium columns
# ---------------------------
# R1-FIX 2 : natremies de TOUTES les lignes de biologie (et non des seules lignes a creatinine numerique)
# Colonnes d'identite directe (nom, prenom, date de naissance) volontairement non chargees : seul l'identifiant interne IPP2 est utilise
na1 = dfglimps_all[['NIP','Date -heure de prvt-','NA  :  Sodium', 'IPP2']].copy()
na1 = na1.rename(columns={'Date -heure de prvt-': 'Date de Prélèvement'})
na1['NAGZA  :  Sodium'] = np.nan
na1['NAGZV  :  Sodium'] = np.nan

na2 = dfglimpsgluc[['NIP','Date de Prélèvement',
                    'NAGZA  :  Sodium','NAGZV  :  Sodium', 'IPP2']].copy()
na2['NA  :  Sodium'] = np.nan  # keep 3-column schema

df_na_final = pd.concat([na1, na2], ignore_index=True)

# types
df_na_final['Date de Prélèvement'] = pd.to_datetime(df_na_final['Date de Prélèvement'], errors='coerce')
for c in ['NA  :  Sodium','NAGZA  :  Sodium','NAGZV  :  Sodium']:
    df_na_final[c] = pd.to_numeric(df_na_final[c], errors='coerce')

# Best raw sodium per row: prioritize NA  :  Sodium, else panel max
df_na_final['NA_raw_best'] = df_na_final['NA  :  Sodium']
panel_max = df_na_final[['NAGZA  :  Sodium','NAGZV  :  Sodium']].max(axis=1, skipna=True)
df_na_final['NA_raw_best'] = df_na_final['NA_raw_best'].fillna(panel_max)

# ---------------------------
# 2) Daily max glycemia (mmol/L) per patient
# ---------------------------
df_glu = dfglimpsgluc[['NIP','Date de Prélèvement','GLGZA  :  Glucose','GLGZV  :  Glucose']].copy()
df_glu['Date de Prélèvement'] = pd.to_datetime(df_glu['Date de Prélèvement'], errors='coerce')
for c in ['GLGZA  :  Glucose','GLGZV  :  Glucose']:
    df_glu[c] = pd.to_numeric(df_glu[c], errors='coerce')

df_glu['glu_row_max_mmolL'] = df_glu[['GLGZA  :  Glucose','GLGZV  :  Glucose']].max(axis=1, skipna=True)
df_glu['day'] = df_glu['Date de Prélèvement'].dt.date

daily_glu = (df_glu.groupby(['NIP','day'], as_index=False)['glu_row_max_mmolL']
             .max()
             .rename(columns={'glu_row_max_mmolL':'glu_daily_max_mmolL'}))

# ---------------------------
# 2bis) Daily max triglycerides (g/L) per patient (fichiers bruts "Raw", colonne TRIG)
# ---------------------------
df_tg = dfglimps_flt[['NIP', 'Date -heure de prvt-', 'TRIG  :  Triglycérides']].copy()
df_tg = df_tg.rename(columns={'Date -heure de prvt-': 'Date de Prélèvement'})
df_tg['Date de Prélèvement'] = pd.to_datetime(df_tg['Date de Prélèvement'], errors='coerce')
df_tg['TRIG_gL'] = pd.to_numeric(
    df_tg['TRIG  :  Triglycérides'].astype(str).str.replace('<', '', regex=False).str.replace(',', '.', regex=False),
    errors='coerce'
)
df_tg['day'] = df_tg['Date de Prélèvement'].dt.date
daily_tg = (df_tg.groupby(['NIP', 'day'], as_index=False)['TRIG_gL'].max()
            .rename(columns={'TRIG_gL': 'tg_daily_max_gL'}))

# ---------------------------
# 3) Merge glycemie + TG, exclusion, puis correction sur NA_raw_best
# ---------------------------
df_na_final['day'] = df_na_final['Date de Prélèvement'].dt.date
df_na_final = df_na_final.merge(daily_glu, on=['NIP','day'], how='left')
df_na_final = df_na_final.merge(daily_tg, on=['NIP', 'day'], how='left')

# Exclusion : glycemie >= 10 mmol/L OU TG >= 2 g/L le meme jour -> valeur non utilisee
major_hyperglycemia = df_na_final['glu_daily_max_mmolL'] >= 10
major_hypertriglyceridemia = df_na_final['tg_daily_max_gL'] >= 2
exclude_mask = major_hyperglycemia.fillna(False) | major_hypertriglyceridemia.fillna(False)
print(f"Natremia values exclues (hyperglycemie majeure >=10 mmol/L ou TG >=2 g/L) : {int(exclude_mask.sum())} / {len(df_na_final)}")
df_na_final.loc[exclude_mask, 'NA_raw_best'] = np.nan

g = df_na_final['glu_daily_max_mmolL']
excess = (g - 5.6).clip(lower=0)

df_na_final['NA_corr_best'] = np.where(
    g.notna() & df_na_final['NA_raw_best'].notna(),
    df_na_final['NA_raw_best'] + 1.6 * (excess / 5.6),
    df_na_final['NA_raw_best']
)


Natremia values exclues (hyperglycemie majeure >=10 mmol/L ou TG >=2 g/L) : 6789 / 114654


## 8. Statistiques de natremie par patient (min/max/mediane/moyenne/premiere/derniere)

In [9]:

df_na_final['NA_corr_best'] = pd.to_numeric(df_na_final['NA_corr_best'], errors='coerce')

#Min
def select_min_natremia(group):
    group = group.dropna(subset=['NA_corr_best'])
    if group.empty:
        return pd.Series([None, None], index=['min_natremia_value', 'min_natremia_date'])
    min_natremia_value = group['NA_corr_best'].min()
    min_natremia_date = group.loc[group['NA_corr_best'].idxmin(), 'Date de Prélèvement']
    return pd.Series([min_natremia_value, min_natremia_date], index=['min_natremia_value', 'min_natremia_date'])

result_min_natremia = df_na_final.groupby('IPP2').apply(select_min_natremia).reset_index()
df = pd.merge(df, result_min_natremia, how='left', on='IPP2')

def time_to_normal_from_min_natremia(group):
    group = group.dropna(subset=['NA_corr_best', 'Date de Prélèvement'])
    if group.empty or group['NA_corr_best'].min() > 135:
        return pd.Series([None, None], index=['time_to_normal_from_min', 'delta_natremia_to_normal_from_min'])
    min_natremia_value = group['NA_corr_best'].min()
    min_natremia_date = group.loc[group['NA_corr_best'].idxmin(), 'Date de Prélèvement']
    group = group[group['Date de Prélèvement'] >= min_natremia_date]
    normal_natremia = group[group['NA_corr_best'] >= 135]
    if normal_natremia.empty:
        return pd.Series([None, min_natremia_value - 135], index=['time_to_normal_from_min', 'delta_natremia_to_normal_from_min'])
    normal_natremia_date = normal_natremia['Date de Prélèvement'].iloc[0]
    time_to_normal_from_min = (normal_natremia_date - min_natremia_date).days
    delta_natremia_to_normal_from_min = min_natremia_value - 135
    return pd.Series([time_to_normal_from_min, delta_natremia_to_normal_from_min], index=['time_to_normal_from_min', 'delta_natremia_to_normal_from_min'])

result_time_to_normal_from_min = df_na_final.groupby('IPP2').apply(time_to_normal_from_min_natremia).reset_index()
df = pd.merge(df, result_time_to_normal_from_min, how='left', on='IPP2')

def time_to_min_natremia_from_normal(group):
    group = group.dropna(subset=['NA_corr_best', 'Date de Prélèvement'])
    if group.empty or group['NA_corr_best'].min() > 135:
        return pd.Series([None, None], index=['time_to_min_from_normal', 'delta_min_natremia_from_normal'])
    min_natremia_value = group['NA_corr_best'].min()
    min_natremia_date = group.loc[group['NA_corr_best'].idxmin(), 'Date de Prélèvement']
    group = group[group['Date de Prélèvement'] <= min_natremia_date]
    normal_natremia = group[group['NA_corr_best'] >= 135]
    if normal_natremia.empty:
        return pd.Series([None, min_natremia_value - 135], index=['time_to_min_from_normal', 'delta_min_natremia_from_normal'])
    normal_natremia_date = normal_natremia['Date de Prélèvement'].iloc[0]
    time_to_min_from_normal = (normal_natremia_date - min_natremia_date).days
    delta_min_natremia_from_normal = min_natremia_value - 135
    return pd.Series([time_to_min_from_normal, delta_min_natremia_from_normal], index=['time_to_min_from_normal', 'delta_min_natremia_from_normal'])

result_time_to_min_from_normal = df_na_final.groupby('IPP2').apply(time_to_min_natremia_from_normal).reset_index()
df = pd.merge(df, result_time_to_min_from_normal, how='left', on='IPP2')

#Max
def select_max_natremia(group):
    group = group.dropna(subset=['NA_corr_best'])
    if group.empty:
        return pd.Series([None, None], index=['max_natremia_value', 'max_natremia_date'])
    max_natremia_value = group['NA_corr_best'].max()
    max_natremia_date = group.loc[group['NA_corr_best'].idxmax(), 'Date de Prélèvement']
    return pd.Series([max_natremia_value, max_natremia_date], index=['max_natremia_value', 'max_natremia_date'])

result_max_natremia = df_na_final.groupby('IPP2').apply(select_max_natremia).reset_index()
df = pd.merge(df, result_max_natremia, how='left', on='IPP2')

def time_to_normal_natremia_from_max(group):
    group = group.dropna(subset=['NA_corr_best', 'Date de Prélèvement'])
    if group.empty or group['NA_corr_best'].max() < 145:
        return pd.Series([None, None], index=['time_to_normal_from_max', 'delta_natremia_from_max'])
    max_natremia_value = group['NA_corr_best'].max()
    max_natremia_date = group.loc[group['NA_corr_best'].idxmax(), 'Date de Prélèvement']
    group = group[group['Date de Prélèvement'] >= max_natremia_date]
    normal_natremia = group[group['NA_corr_best'] <= 145]
    if normal_natremia.empty:
        return pd.Series([None, max_natremia_value - 145], index=['time_to_normal_from_max', 'delta_natremia_from_max'])
    normal_natremia_date = normal_natremia['Date de Prélèvement'].iloc[0]
    time_to_normal_from_max = (normal_natremia_date - max_natremia_date).days
    delta_natremia_from_max = max_natremia_value - 145
    return pd.Series([time_to_normal_from_max, delta_natremia_from_max], index=['time_to_normal_from_max', 'delta_natremia_from_max'])

result_time_to_normal_from_max = df_na_final.groupby('IPP2').apply(time_to_normal_natremia_from_max).reset_index()
df = pd.merge(df, result_time_to_normal_from_max, how='left', on='IPP2')

def time_to_max_natremia_from_normal(group):
    group = group.dropna(subset=['NA_corr_best', 'Date de Prélèvement'])
    if group.empty or group['NA_corr_best'].max() < 145:
        return pd.Series([None, None], index=['time_to_max_from_normal', 'delta_max_natremia_from_normal'])
    max_natremia_value = group['NA_corr_best'].max()
    max_natremia_date = group.loc[group['NA_corr_best'].idxmax(), 'Date de Prélèvement']
    group = group[group['Date de Prélèvement'] <= max_natremia_date]
    normal_natremia = group[group['NA_corr_best'] <= 145]
    if normal_natremia.empty:
        return pd.Series([None, max_natremia_value - 145], index=['time_to_max_from_normal', 'delta_max_natremia_from_normal'])
    normal_natremia_date = normal_natremia['Date de Prélèvement'].iloc[0]
    time_to_max_from_normal = (normal_natremia_date - max_natremia_date).days
    delta_max_natremia_from_normal = max_natremia_value - 145
    return pd.Series([time_to_max_from_normal, delta_max_natremia_from_normal], index=['time_to_max_from_normal', 'delta_max_natremia_from_normal'])

result_time_to_max_from_normal = df_na_final.groupby('IPP2').apply(time_to_max_natremia_from_normal).reset_index()
df = pd.merge(df, result_time_to_max_from_normal, how='left', on='IPP2')

#median
def select_median_natremia(group):
    group = group.dropna(subset=['NA_corr_best'])
    if group.empty:
        return pd.Series([None], index=['median_natremia_value'])
    return pd.Series([group['NA_corr_best'].median()], index=['median_natremia_value'])

result_median_natremia = df_na_final.groupby('IPP2').apply(select_median_natremia).reset_index()
df = pd.merge(df, result_median_natremia, how='left', on='IPP2')

#mean
def select_mean_natremia(group):
    group = group.dropna(subset=['NA_corr_best'])
    if group.empty:
        return pd.Series([None], index=['mean_natremia_value'])
    return pd.Series([group['NA_corr_best'].mean()], index=['mean_natremia_value'])

result_mean_natremia = df_na_final.groupby('IPP2').apply(select_mean_natremia).reset_index()
df = pd.merge(df, result_mean_natremia, how='left', on='IPP2')

def select_first_natremia(group):
    group = group.dropna(subset=['NA_corr_best', 'Date de Prélèvement'])
    group = group.sort_values(by='Date de Prélèvement')
    if group.empty:
        return pd.Series([None, None], index=['first_natremia_value', 'first_natremia_date'])
    return pd.Series([group['NA_corr_best'].iloc[0], group['Date de Prélèvement'].iloc[0]], index=['first_natremia_value', 'first_natremia_date'])

result_first_natremia = df_na_final.groupby('IPP2').apply(select_first_natremia).reset_index()
df = pd.merge(df, result_first_natremia, how='left', on='IPP2')

def select_last_natremia(group):
    group = group.dropna(subset=['NA_corr_best', 'Date de Prélèvement'])
    group = group.sort_values(by='Date de Prélèvement')
    if group.empty:
        return pd.Series([None, None], index=['last_natremia_value', 'last_natremia_date'])
    return pd.Series([group['NA_corr_best'].iloc[-1], group['Date de Prélèvement'].iloc[-1]], index=['last_natremia_value', 'last_natremia_date'])

result_last_natremia = df_na_final.groupby('IPP2').apply(select_last_natremia).reset_index()
df = pd.merge(df, result_last_natremia, how='left', on='IPP2')

print("min_natremia_value :", df['min_natremia_value'].describe()[['mean','min','max']].to_dict())
print("max_natremia_value :", df['max_natremia_value'].describe()[['mean','min','max']].to_dict())


min_natremia_value : {'mean': 134.20004116920543, 'min': 72.5, 'max': 146.7}
max_natremia_value : {'mean': 143.26824207492797, 'min': 130.1, 'max': 197.2}


## 9. Premiere date d'hyper/hyponatremie

In [10]:

df_na_final['NA_corr_best'] = pd.to_numeric(df_na_final['NA_corr_best'], errors='coerce')
df_na_final['Date de Prélèvement'] = pd.to_datetime(df_na_final['Date de Prélèvement'], format='%d-%m-%y %Hh%Mmin%Ss')


df_na_final = df_na_final.sort_values(['IPP2', 'Date de Prélèvement']).reset_index(drop=True)   # R1-FIX 4

def first_natremia_above_145(group):
    group = group.dropna(subset=['NA_corr_best', 'Date de Prélèvement'])
    if group.empty or group['NA_corr_best'].max() <= 145:
        return pd.Series([None], index=['first_natremia_above_145_date'])
    above_145 = group[group['NA_corr_best'] > 145]
    if above_145.empty:
        return pd.Series([None], index=['first_natremia_above_145_date'])
    return pd.Series([above_145['Date de Prélèvement'].iloc[0]], index=['first_natremia_above_145_date'])

result_1st_hypernatremia = df_na_final.groupby('IPP2').apply(first_natremia_above_145).reset_index()
df = pd.merge(df, result_1st_hypernatremia, how='left', on='IPP2')

def first_natremia_below_135(group):
    group = group.dropna(subset=['NA_corr_best', 'Date de Prélèvement'])
    if group.empty or group['NA_corr_best'].min() >= 135:
        return pd.Series([None], index=['first_natremia_below_135_date'])
    below_135 = group[group['NA_corr_best'] < 135]
    if below_135.empty:
        return pd.Series([None], index=['first_natremia_below_135_date'])
    return pd.Series([below_135['Date de Prélèvement'].iloc[0]], index=['first_natremia_below_135_date'])

result_1st_hyponatremia = df_na_final.groupby('IPP2').apply(first_natremia_below_135).reset_index()
df = pd.merge(df, result_1st_hyponatremia, how='left', on='IPP2')

df['first_natremia_above_145_date'] = pd.to_datetime(df['first_natremia_above_145_date'], errors='coerce')
df['first_natremia_below_135_date'] = pd.to_datetime(df['first_natremia_below_135_date'], errors='coerce')


## 10. Nombre de jours en hypo/hypernatremie

In [11]:

def calculate_days_with_natremia(group):
    group['NA_corr_best'] = pd.to_numeric(group['NA_corr_best'], errors='coerce')
    group = group.sort_values(by='Date de Prélèvement', ascending=True)
    if group.empty:
        return pd.Series([None, None], index=['days_na_above_145', 'days_na_below_135'])

    group_above_145 = group[group['NA_corr_best'] > 145]
    days_above_145 = (group_above_145['Date de Prélèvement'].max() - group_above_145['Date de Prélèvement'].min()).days if not group_above_145.empty else 0

    group_below_135 = group[group['NA_corr_best'] < 135]
    days_below_135 = (group_below_135['Date de Prélèvement'].max() - group_below_135['Date de Prélèvement'].min()).days if not group_below_135.empty else 0

    return pd.Series([days_above_145, days_below_135], index=['days_na_above_145', 'days_na_below_135'])

result_days_natremia = df_na_final.groupby('IPP2').apply(calculate_days_with_natremia).reset_index()
df = pd.merge(df, result_days_natremia, how='left', on='IPP2')


## 11. Episodes de dysnatremie (comptage + dates)

In [12]:

def dysnatremia_episodes_with_dates(group, col_na='NA_corr_best', col_date='Date de Prélèvement', hypo_thr=135, hyper_thr=145):
    g = group.dropna(subset=[col_na, col_date]).sort_values(col_date).copy()
    if g.empty:
        return pd.Series({'n_hypo_episodes': 0, 'first_hypo_date': pd.NaT, 'n_hyper_episodes': 0, 'first_hyper_date': pd.NaT})

    na = g[col_na].values
    dates = g[col_date].values
    n_hypo = n_hyper = 0
    first_hypo_date = pd.NaT
    first_hyper_date = pd.NaT
    in_hypo = in_hyper = False

    for i in range(len(na)):
        if na[i] < hypo_thr:
            if not in_hypo:
                n_hypo += 1
                in_hypo = True
                if pd.isna(first_hypo_date):
                    first_hypo_date = dates[i]
        else:
            in_hypo = False

        if na[i] > hyper_thr:
            if not in_hyper:
                n_hyper += 1
                in_hyper = True
                if pd.isna(first_hyper_date):
                    first_hyper_date = dates[i]
        else:
            in_hyper = False

    return pd.Series({'n_hypo_episodes': n_hypo, 'first_hypo_date': first_hypo_date, 'n_hyper_episodes': n_hyper, 'first_hyper_date': first_hyper_date})

episodes = df_na_final.groupby('IPP2').apply(dysnatremia_episodes_with_dates).reset_index()
df = df.merge(episodes, on='IPP2', how='left')


## 12. Table detaillee des episodes (format long)

In [13]:

def dysnatremia_episode_table(group, id_col='IPP2', col_na='NA_corr_best', col_date='Date de Prélèvement', hypo_thr=135, hyper_thr=145):
    g = (
        group[[id_col, col_na, col_date]]
        .dropna(subset=[col_na, col_date])
        .sort_values(col_date)
        .reset_index(drop=True)
        .copy()
    )
    if g.empty:
        return pd.DataFrame(columns=[id_col, 'episode_type', 'start_date', 'end_date', 'n_points', 'na_min', 'na_max'])

    na = g[col_na].to_numpy()
    dt = pd.to_datetime(g[col_date]).to_numpy()
    rows = []

    def extract_runs(mask, episode_type):
        in_run = False
        start_i = None
        for i, flag in enumerate(mask):
            if flag and not in_run:
                in_run = True
                start_i = i
            is_last = (i == len(mask) - 1)
            if in_run and ((not flag) or is_last):
                end_i = i if (flag and is_last) else (i - 1)
                rows.append({
                    id_col: g.loc[0, id_col],
                    'episode_type': episode_type,
                    'start_date': dt[start_i],
                    'end_date': dt[end_i],
                    'n_points': int(end_i - start_i + 1),
                    'na_min': float(np.nanmin(na[start_i:end_i+1])),
                    'na_max': float(np.nanmax(na[start_i:end_i+1])),
                })
                in_run = False
                start_i = None

    extract_runs(na < hypo_thr, 'hypo')
    extract_runs(na > hyper_thr, 'hyper')
    return pd.DataFrame(rows)

episodes_long = (
    df_na_final
    .groupby('IPP2', group_keys=False)
    .apply(dysnatremia_episode_table)
    .reset_index(drop=True)
)
print("Episodes (format long) :", episodes_long.shape)


Episodes (format long) : (5480, 7)


## 13. Exposition cumulee (AUC) a l'hypo/hypernatremie

In [14]:

df_na_final['NA_corr_best'] = pd.to_numeric(df_na_final['NA_corr_best'], errors='coerce')
df_na_final['Date de Prélèvement'] = pd.to_datetime(df_na_final['Date de Prélèvement'])

def auc_excess_linear(t0, y0, t1, y1, thr, above=True):
    if t1 <= t0 or np.isnan(y0) or np.isnan(y1):
        return 0.0
    dt = t1 - t0
    if above:
        a0, a1 = y0 - thr, y1 - thr
    else:
        a0, a1 = thr - y0, thr - y1
    if a0 <= 0 and a1 <= 0:
        return 0.0
    if a0 >= 0 and a1 >= 0:
        return 0.5 * (a0 + a1) * dt
    frac = a0 / (a0 - a1)
    tc = t0 + frac * dt
    if a0 > 0 and a1 < 0:
        return 0.5 * (a0 + 0.0) * (tc - t0)
    else:
        return 0.5 * (0.0 + a1) * (t1 - tc)

def compute_natremia_auc(group, col_na='NA_corr_best', col_date='Date de Prélèvement', thr_low=135, thr_high=145, time_unit='days'):
    g = group.dropna(subset=[col_na, col_date]).sort_values(col_date).copy()
    if len(g) < 2:
        return pd.Series({'AUC_na_above145': np.nan, 'AUC_na_below135': np.nan})
    t = (g[col_date] - g[col_date].iloc[0]).dt.total_seconds()
    t = t / (3600.0 * 24.0) if time_unit == 'days' else t / 3600.0
    y = g[col_na].astype(float).to_numpy()
    t = t.to_numpy()
    auc_high = auc_low = 0.0
    for i in range(len(y) - 1):
        auc_high += auc_excess_linear(t[i], y[i], t[i+1], y[i+1], thr_high, above=True)
        auc_low += auc_excess_linear(t[i], y[i], t[i+1], y[i+1], thr_low, above=False)
    return pd.Series({'AUC_na_above145': auc_high, 'AUC_na_below135': auc_low})

nat_auc = df_na_final.groupby('IPP2').apply(compute_natremia_auc).reset_index()
df = df.merge(nat_auc, on='IPP2', how='left')


## 14. Glycemie (labo + dextros capillaires)

In [15]:

# R1-FIX 5 : import sklearn inutilise supprime

df_hgt_cappilary = pd.read_csv(f'{DATA}/HGT_natremie.csv', sep=';')
df_hgt_cappilary['valeur'] = (df_hgt_cappilary['valeur']/100)

df_glycemia = pd.concat([
    dfglimpsgluc[['IPP2', 'Date de Prélèvement', 'GLGZA  :  Glucose']].rename(columns={'GLGZA  :  Glucose': 'Glucose'}),
    dfglimpsgluc[['IPP2', 'Date de Prélèvement', 'GLGZV  :  Glucose']].rename(columns={'GLGZV  :  Glucose': 'Glucose'}),
    df_hgt_cappilary.rename(columns={'IPP': 'IPP2', 'valeur': 'Glucose'})
], ignore_index=True)

df_glycemia['IPP2'] = pd.to_numeric(df_glycemia['IPP2'], errors='coerce')
df_glycemia['Glucose'] = pd.to_numeric(df_glycemia['Glucose'], errors='coerce')
df_glycemia = df_glycemia.dropna(subset=['Glucose'])

result_median_glucose = df_glycemia.groupby('IPP2', as_index=False)['Glucose'].median().rename(columns={'Glucose': 'median_glucose_value'})
df = df.merge(result_median_glucose, on='IPP2', how='left')

result_min_glucose = df_glycemia.groupby('IPP2', as_index=False)['Glucose'].min().rename(columns={'Glucose': 'min_glucose_value'})
df = df.merge(result_min_glucose, on='IPP2', how='left')

result_max_glucose = df_glycemia.groupby('IPP2', as_index=False)['Glucose'].max().rename(columns={'Glucose': 'max_glucose_value'})
df = df.merge(result_max_glucose, on='IPP2', how='left')


## 15. Uree (median/min/max)

In [16]:

dfglimps_flt['UR  :  Urée'] = pd.to_numeric(dfglimps_flt['UR  :  Urée'], errors='coerce')

def select_median_urea(group):
    group = group.dropna(subset=['UR  :  Urée'])
    if group.empty:
        return pd.Series([None], index=['median_urea_value'])
    return pd.Series([group['UR  :  Urée'].median()], index=['median_urea_value'])

result_median_urea = dfglimps_flt.groupby('IPP2').apply(select_median_urea).reset_index()
df = pd.merge(df, result_median_urea, how='left', on='IPP2')

def select_min_urea(group):
    group = group.dropna(subset=['UR  :  Urée'])
    if group.empty:
        return pd.Series([None], index=['min_urea_value'])
    return pd.Series([group['UR  :  Urée'].min()], index=['min_urea_value'])

result_min_urea = dfglimps_flt.groupby('IPP2').apply(select_min_urea).reset_index()
df = pd.merge(df, result_min_urea, how='left', on='IPP2')

def select_max_urea(group):
    group = group.dropna(subset=['UR  :  Urée'])
    if group.empty:
        return pd.Series([None], index=['max_urea_value'])
    return pd.Series([group['UR  :  Urée'].max()], index=['max_urea_value'])

result_max_urea = dfglimps_flt.groupby('IPP2').apply(select_max_urea).reset_index()
df = pd.merge(df, result_max_urea, how='left', on='IPP2')


## 16. Biologie urinaire journaliere (osmolarite estimee)

In [17]:

dfglimpsurine['day'] = dfglimpsurine['Date de Prélèvement'].dt.floor('D')

daily_urine = (
    dfglimpsurine
    .groupby(['NIP', 'day'], as_index=False)[urine_cols]
    .mean()
)

daily_urine['Uosm_est'] = 2 * (daily_urine['NAU  :  Sodium'] + daily_urine['KU  :  Potassium']) + daily_urine['URU  :  Urée']
daily_urine['Uosm_gt_100'] = (daily_urine['Uosm_est'] > 100).astype('int')
daily_urine['UNa_gt_30'] = (daily_urine['NAU  :  Sodium'] > 30).astype('int')


## 17. Episodes d'hyponatremie : biologie urinaire + diuretiques associes

In [18]:

df_loop = df_loop.rename(columns={'IPP':'IPP2'})

daily_urine_df = daily_urine.rename(columns={'NIP': 'IPP2'}).copy()
daily_urine_df['day'] = pd.to_datetime(daily_urine_df['day'], errors='coerce').dt.floor('D')

df_na_final_df = df_na_final.copy()
df_na_final_df['Date de Prélèvement'] = pd.to_datetime(df_na_final_df['Date de Prélèvement'], errors='coerce')
df_na_final_df['NA_corr_best'] = pd.to_numeric(df_na_final_df['NA_corr_best'], errors='coerce')

for data in [df_na_final_df, daily_urine_df, df_loop]:
    data['IPP2'] = pd.to_numeric(data['IPP2'], errors='coerce')

dfd = df_loop.copy()
dfd['day'] = pd.to_datetime(dfd['day'], errors='coerce').dt.floor('D')

diur_daily = (
    dfd.dropna(subset=['IPP2', 'day'])
       .assign(n_diur_admin=1)
       .groupby(['IPP2', 'day'], as_index=False)['n_diur_admin'].sum()
)

def label_dysnatremia_episodes(group, col_na='NA_corr_best', col_date='Date de Prélèvement', hypo_thr=135, hyper_thr=145):
    g = group.dropna(subset=[col_na, col_date]).sort_values(col_date).copy()
    if g.empty:
        g['hypo_ep_id'] = np.nan
        g['hyper_ep_id'] = np.nan
        return g

    na = pd.to_numeric(g[col_na], errors='coerce').to_numpy()
    hypo_id = hyper_id = 0
    in_hypo = in_hyper = False
    hypo_ep_ids, hyper_ep_ids = [], []

    for v in na:
        if v < hypo_thr:
            if not in_hypo:
                hypo_id += 1
                in_hypo = True
            hypo_ep_ids.append(hypo_id)
        else:
            in_hypo = False
            hypo_ep_ids.append(np.nan)

        if v > hyper_thr:
            if not in_hyper:
                hyper_id += 1
                in_hyper = True
            hyper_ep_ids.append(hyper_id)
        else:
            in_hyper = False
            hyper_ep_ids.append(np.nan)

    g['hypo_ep_id'] = hypo_ep_ids
    g['hyper_ep_id'] = hyper_ep_ids
    return g

df_na_labeled = df_na_final_df.groupby('IPP2', group_keys=False).apply(label_dysnatremia_episodes)
df_na_labeled['day'] = df_na_labeled['Date de Prélèvement'].dt.floor('D')

hypo_windows = (
    df_na_labeled.dropna(subset=['hypo_ep_id'])
    .groupby(['IPP2', 'hypo_ep_id'], as_index=False)
    .agg(hypo_start_day=('day', 'min'), hypo_end_day=('day', 'max'))
)

def expand_days(windows, start_col, end_col, ep_col):
    tmp = windows.copy()
    tmp['day'] = tmp.apply(lambda r: pd.date_range(r[start_col], r[end_col], freq='D'), axis=1)
    return tmp.explode('day')[['IPP2', ep_col, 'day']]

hypo_days = expand_days(hypo_windows, 'hypo_start_day', 'hypo_end_day', ep_col='hypo_ep_id')
hypo_urine = hypo_days.merge(daily_urine_df, on=['IPP2', 'day'], how='left')

tmp_first = (
    hypo_urine
      .sort_values(['IPP2', 'hypo_ep_id', 'day'])
      .assign(UNa_num=lambda d: pd.to_numeric(d['NAU  :  Sodium'], errors='coerce'))
      .dropna(subset=['IPP2', 'hypo_ep_id', 'UNa_num'])
)

first_una = (
    tmp_first.groupby(['IPP2', 'hypo_ep_id'], as_index=False)
             .first()[['IPP2', 'hypo_ep_id', 'day', 'UNa_num']]
             .rename(columns={'day': 'first_UNa_day', 'UNa_num': 'first_UNa'})
)

def _any_nonnull(s):
    return int(pd.to_numeric(s, errors='coerce').notna().any())

for c in ['NAU  :  Sodium', 'Uosm_est']:
    if c in daily_urine_df.columns:
        daily_urine_df[c] = (
            daily_urine_df[c].astype(str)
            .str.replace('<', '', regex=False)
            .str.replace(',', '.', regex=False)
        )
        daily_urine_df[c] = pd.to_numeric(daily_urine_df[c], errors='coerce')

urine_stats = (
    hypo_urine.groupby(['IPP2','hypo_ep_id'], as_index=False)
    .agg(
        min_UNa=('NAU  :  Sodium', 'min'),
        max_UNa=('NAU  :  Sodium', 'max'),
        min_Uosm=('Uosm_est', 'min'),
        max_Uosm=('Uosm_est', 'max'),
        any_UNa_lt30=('NAU  :  Sodium', lambda x: int((pd.to_numeric(x, errors='coerce') < 30).any())),
        any_Uosm_gt100=('Uosm_est', lambda x: int((pd.to_numeric(x, errors='coerce') > 100).any())),
        has_UNa=('NAU  :  Sodium', _any_nonnull),
        has_Uosm=('Uosm_est', _any_nonnull),
    )
)

hypo_episode_urine_summary = (
    hypo_windows.merge(urine_stats, on=['IPP2','hypo_ep_id'], how='left')
    .rename(columns={'hypo_start_day':'start', 'hypo_end_day':'end'})
)
hypo_episode_urine_summary = hypo_episode_urine_summary.merge(first_una, on=['IPP2', 'hypo_ep_id'], how='left')
hypo_episode_urine_summary[['has_UNa','has_Uosm']] = hypo_episode_urine_summary[['has_UNa','has_Uosm']].fillna(0).astype(int)
hypo_episode_urine_summary['no_urine_data'] = (
    (hypo_episode_urine_summary['has_UNa'] == 0) & (hypo_episode_urine_summary['has_Uosm'] == 0)
).astype(int)

# Diuretiques J-1 / pendant / J+1
tmp = hypo_episode_urine_summary[['IPP2','hypo_ep_id','start','end']].copy()
tmp['day_before'] = tmp['start'] - pd.Timedelta(days=1)
before = tmp.merge(diur_daily, left_on=['IPP2','day_before'], right_on=['IPP2','day'], how='left').rename(columns={'n_diur_admin':'n_diur_before'})
tmp['day_after'] = tmp['end'] + pd.Timedelta(days=1)
after = tmp.merge(diur_daily, left_on=['IPP2','day_after'], right_on=['IPP2','day'], how='left').rename(columns={'n_diur_admin':'n_diur_after'})

hypo_days_for_diur = expand_days(
    hypo_episode_urine_summary.rename(columns={'start':'hypo_start_day','end':'hypo_end_day'}),
    'hypo_start_day', 'hypo_end_day', ep_col='hypo_ep_id'
)
during = hypo_days_for_diur.merge(diur_daily, on=['IPP2','day'], how='left')
during_stats = during.groupby(['IPP2','hypo_ep_id'], as_index=False).agg(n_diur_during=('n_diur_admin','sum'))

hypo_episode_urine_summary = (
    hypo_episode_urine_summary
    .merge(before[['IPP2','hypo_ep_id','n_diur_before']], on=['IPP2','hypo_ep_id'], how='left')
    .merge(during_stats, on=['IPP2','hypo_ep_id'], how='left')
    .merge(after[['IPP2','hypo_ep_id','n_diur_after']], on=['IPP2','hypo_ep_id'], how='left')
)

for c in ['n_diur_before','n_diur_during','n_diur_after']:
    hypo_episode_urine_summary[c] = pd.to_numeric(hypo_episode_urine_summary[c], errors='coerce').fillna(0).astype(int)

hypo_episode_urine_summary['diur_before_24h'] = (hypo_episode_urine_summary['n_diur_before'] > 0).astype(int)
hypo_episode_urine_summary['diur_during']     = (hypo_episode_urine_summary['n_diur_during'] > 0).astype(int)
hypo_episode_urine_summary['diur_after_24h']  = (hypo_episode_urine_summary['n_diur_after'] > 0).astype(int)


## 18. Episodes d'hypernatremie : biologie urinaire + diuretiques associes

In [19]:

hyper_windows = (
    df_na_labeled.dropna(subset=['hyper_ep_id'])
    .groupby(['IPP2', 'hyper_ep_id'], as_index=False)
    .agg(hyper_start_day=('day', 'min'), hyper_end_day=('day', 'max'))
)

hyper_days = expand_days(hyper_windows, 'hyper_start_day', 'hyper_end_day', ep_col='hyper_ep_id')
hyper_urine = hyper_days.merge(daily_urine_df, on=['IPP2', 'day'], how='left')

hyper_urine_stats = (
    hyper_urine.groupby(['IPP2', 'hyper_ep_id'], as_index=False)
    .agg(
        min_UNa=('NAU  :  Sodium', 'min'),
        max_UNa=('NAU  :  Sodium', 'max'),
        min_Uosm=('Uosm_est', 'min'),
        max_Uosm=('Uosm_est', 'max'),
        any_UNa_lt30=('NAU  :  Sodium', lambda x: int((pd.to_numeric(x, errors='coerce') < 30).any())),
        any_Uosm_gt100=('Uosm_est', lambda x: int((pd.to_numeric(x, errors='coerce') > 100).any())),
        has_UNa=('NAU  :  Sodium', _any_nonnull),
        has_Uosm=('Uosm_est', _any_nonnull),
    )
)

hyper_episode_urine_summary = (
    hyper_windows.merge(hyper_urine_stats, on=['IPP2', 'hyper_ep_id'], how='left')
    .rename(columns={'hyper_start_day': 'start', 'hyper_end_day': 'end'})
)
hyper_episode_urine_summary[['has_UNa', 'has_Uosm']] = hyper_episode_urine_summary[['has_UNa', 'has_Uosm']].fillna(0).astype(int)
hyper_episode_urine_summary['no_urine_data'] = (
    (hyper_episode_urine_summary['has_UNa'] == 0) & (hyper_episode_urine_summary['has_Uosm'] == 0)
).astype(int)

tmp = hyper_episode_urine_summary[['IPP2', 'hyper_ep_id', 'start', 'end']].copy()
tmp['day_before'] = tmp['start'] - pd.Timedelta(days=1)
before = tmp.merge(diur_daily, left_on=['IPP2', 'day_before'], right_on=['IPP2', 'day'], how='left').rename(columns={'n_diur_admin': 'n_diur_before'})
tmp['day_after'] = tmp['end'] + pd.Timedelta(days=1)
after = tmp.merge(diur_daily, left_on=['IPP2', 'day_after'], right_on=['IPP2', 'day'], how='left').rename(columns={'n_diur_admin': 'n_diur_after'})

hyper_days_for_diur = expand_days(
    hyper_episode_urine_summary.rename(columns={'start': 'hyper_start_day', 'end': 'hyper_end_day'}),
    'hyper_start_day', 'hyper_end_day', ep_col='hyper_ep_id'
)
during = hyper_days_for_diur.merge(diur_daily, on=['IPP2', 'day'], how='left')
during['n_diur_admin'] = pd.to_numeric(during['n_diur_admin'], errors='coerce').fillna(0)
during_stats = during.groupby(['IPP2', 'hyper_ep_id'], as_index=False).agg(n_diur_during=('n_diur_admin', 'sum'))

hyper_episode_urine_summary = (
    hyper_episode_urine_summary
    .merge(before[['IPP2', 'hyper_ep_id', 'n_diur_before']], on=['IPP2', 'hyper_ep_id'], how='left')
    .merge(during_stats, on=['IPP2', 'hyper_ep_id'], how='left')
    .merge(after[['IPP2', 'hyper_ep_id', 'n_diur_after']], on=['IPP2', 'hyper_ep_id'], how='left')
)

for c in ['n_diur_before', 'n_diur_during', 'n_diur_after']:
    hyper_episode_urine_summary[c] = pd.to_numeric(hyper_episode_urine_summary[c], errors='coerce').fillna(0).astype(int)

hyper_episode_urine_summary['diur_before_24h'] = (hyper_episode_urine_summary['n_diur_before'] > 0).astype(int)
hyper_episode_urine_summary['diur_during']     = (hyper_episode_urine_summary['n_diur_during'] > 0).astype(int)
hyper_episode_urine_summary['diur_after_24h']  = (hyper_episode_urine_summary['n_diur_after'] > 0).astype(int)


## 19. Apports hydriques autour des episodes (hypo)

In [20]:

df_inputs_df = df_inputs.rename(columns={'IPP': 'IPP2'}).copy()
df_inputs_df['IPP2'] = pd.to_numeric(df_inputs_df['IPP2'], errors='coerce')
df_inputs_df['day']  = pd.to_datetime(df_inputs_df['day'], errors='coerce').dt.floor('D')
df_inputs_df['in_ml'] = pd.to_numeric(df_inputs_df['in_ml'], errors='coerce')

intake_daily = (
    df_inputs_df.dropna(subset=['IPP2', 'day'])
        .groupby(['IPP2', 'day'], as_index=False)['in_ml']
        .sum()
)

hypo_windows2 = hypo_windows.rename(columns={'hypo_start_day': 'start', 'hypo_end_day': 'end'}).copy()

tmp = hypo_windows2[['IPP2', 'hypo_ep_id', 'start', 'end']].copy()
tmp['day_before'] = tmp['start'] - pd.Timedelta(days=1)
before = tmp.merge(intake_daily, left_on=['IPP2', 'day_before'], right_on=['IPP2', 'day'], how='left').rename(columns={'in_ml': 'mean_in_ml_before'})
tmp['day_after'] = tmp['end'] + pd.Timedelta(days=1)
after = tmp.merge(intake_daily, left_on=['IPP2', 'day_after'], right_on=['IPP2', 'day'], how='left').rename(columns={'in_ml': 'mean_in_ml_after'})

hypo_days_for_intake = expand_days(
    hypo_windows2.rename(columns={'start': 'hypo_start_day', 'end': 'hypo_end_day'}),
    'hypo_start_day', 'hypo_end_day', ep_col='hypo_ep_id'
)
during = hypo_days_for_intake.merge(intake_daily, on=['IPP2', 'day'], how='left')
during['in_ml'] = pd.to_numeric(during['in_ml'], errors='coerce')

during_stats = (
    during.groupby(['IPP2', 'hypo_ep_id'], as_index=False)
          .agg(mean_in_ml_during=('in_ml', 'mean'), n_days_with_intake=('in_ml', lambda x: int(x.notna().sum())), n_days_episode=('day', 'nunique'))
)

hypo_episode_input_summary = (
    hypo_windows2
    .merge(before[['IPP2', 'hypo_ep_id', 'mean_in_ml_before']], on=['IPP2', 'hypo_ep_id'], how='left')
    .merge(during_stats, on=['IPP2', 'hypo_ep_id'], how='left')
    .merge(after[['IPP2', 'hypo_ep_id', 'mean_in_ml_after']], on=['IPP2', 'hypo_ep_id'], how='left')
)


## 20. Apports hydriques autour des episodes (hyper)

In [21]:

hyper_windows2 = (
    hyper_windows.rename(columns={'hyper_start_day': 'start', 'hyper_end_day': 'end'}).copy()
    if 'hyper_start_day' in hyper_windows.columns else hyper_windows.copy()
)

tmp = hyper_windows2[['IPP2', 'hyper_ep_id', 'start', 'end']].copy()
tmp['day_before'] = tmp['start'] - pd.Timedelta(days=1)
before = tmp.merge(intake_daily, left_on=['IPP2', 'day_before'], right_on=['IPP2', 'day'], how='left').rename(columns={'in_ml': 'mean_in_ml_before'})
tmp['day_after'] = tmp['end'] + pd.Timedelta(days=1)
after = tmp.merge(intake_daily, left_on=['IPP2', 'day_after'], right_on=['IPP2', 'day'], how='left').rename(columns={'in_ml': 'mean_in_ml_after'})

hyper_days_for_intake = expand_days(
    hyper_windows2.rename(columns={'start': 'hyper_start_day', 'end': 'hyper_end_day'}),
    'hyper_start_day', 'hyper_end_day', ep_col='hyper_ep_id'
)
during = hyper_days_for_intake.merge(intake_daily, on=['IPP2', 'day'], how='left')
during['in_ml'] = pd.to_numeric(during['in_ml'], errors='coerce')

during_stats = (
    during.groupby(['IPP2', 'hyper_ep_id'], as_index=False)
          .agg(mean_in_ml_during=('in_ml', 'mean'), n_days_with_intake=('in_ml', lambda x: int(x.notna().sum())), n_days_episode=('day', 'nunique'))
)

hyper_episode_input_summary = (
    hyper_windows2
    .merge(before[['IPP2', 'hyper_ep_id', 'mean_in_ml_before']], on=['IPP2', 'hyper_ep_id'], how='left')
    .merge(during_stats, on=['IPP2', 'hyper_ep_id'], how='left')
    .merge(after[['IPP2', 'hyper_ep_id', 'mean_in_ml_after']], on=['IPP2', 'hyper_ep_id'], how='left')
)


## 21. Tables episode complet (biologie urinaire + apports + diuretiques)

In [22]:

hypo_episode_full = hypo_episode_input_summary.merge(
    hypo_episode_urine_summary, on=['IPP2', 'hypo_ep_id'], how='outer', suffixes=('_input', '_urine')
)

hyper_episode_full = hyper_episode_input_summary.merge(
    hyper_episode_urine_summary, on=['IPP2', 'hyper_ep_id'], how='outer', suffixes=('_input', '_urine')
)

print("hypo_episode_full  :", hypo_episode_full.shape)
print("hyper_episode_full :", hyper_episode_full.shape)


hypo_episode_full  : (3274, 28)
hyper_episode_full : (2206, 26)


## 22. Osmolarite calculee

In [23]:

df['osmo_calc'] = df['median_natremia_value']*2 + df['median_glucose_value'] + df['median_urea_value']


## 23. Remplissage vasculaire des premieres 48h

In [24]:

DF_IPP_COL = 'IPP2'
BURN_DATE_COL = 'D.burn'
INPUT_IPP_COL = 'IPP'
INPUT_DAY_COL = 'day'
INPUT_IN_COL = 'in_ml'

df2 = df.copy()
inp = df_inputs.copy()

if INPUT_IPP_COL != 'IPP2':
    inp = inp.rename(columns={INPUT_IPP_COL: 'IPP2'})
if DF_IPP_COL != 'IPP2':
    df2 = df2.rename(columns={DF_IPP_COL: 'IPP2'})

df2['IPP2'] = df2['IPP2'].astype(str)
inp['IPP2'] = inp['IPP2'].astype(str)

df2[BURN_DATE_COL] = pd.to_datetime(df2[BURN_DATE_COL], errors='coerce').dt.floor('D')
inp[INPUT_DAY_COL] = pd.to_datetime(inp[INPUT_DAY_COL], errors='coerce').dt.floor('D')
inp[INPUT_IN_COL] = pd.to_numeric(inp[INPUT_IN_COL], errors='coerce')

inp_daily = (
    inp.dropna(subset=['IPP2', INPUT_DAY_COL])
       .groupby(['IPP2', INPUT_DAY_COL], as_index=False)[INPUT_IN_COL]
       .sum()
       .rename(columns={INPUT_DAY_COL: 'day', INPUT_IN_COL: 'in_ml'})
)

j = df2[['IPP2', BURN_DATE_COL]].merge(inp_daily, on='IPP2', how='left')
j = j.loc[(j['day'] >= j[BURN_DATE_COL]) & (j['in_ml'].notna())].copy()

first_day = j.groupby(['IPP2', BURN_DATE_COL], as_index=False).agg(first_intake_day=('day', 'min'))
df2 = df2.merge(first_day, on=['IPP2', BURN_DATE_COL], how='left')

j2 = df2[['IPP2', BURN_DATE_COL, 'first_intake_day']].merge(inp_daily, on='IPP2', how='left')
mask48 = (
    j2['first_intake_day'].notna() &
    (j2['day'] >= j2['first_intake_day']) &
    (j2['day'] < j2['first_intake_day'] + pd.Timedelta(days=2))
)

fluid_48h = (
    j2.loc[mask48]
      .groupby(['IPP2', BURN_DATE_COL], as_index=False)['in_ml']
      .sum()
      .rename(columns={'in_ml': 'fluid_in_ml_first_48h'})
)

df2 = df2.merge(fluid_48h, on=['IPP2', BURN_DATE_COL], how='left')
df2['fluid_in_ml_first_48h'] = pd.to_numeric(df2['fluid_in_ml_first_48h'], errors='coerce').fillna(0)

if DF_IPP_COL != 'IPP2':
    df2 = df2.rename(columns={'IPP2': DF_IPP_COL})


## 24. Deduplication et export final

In [25]:

df2 = df2.drop_duplicates(subset='IPP2')

# Patients sans aucune mesure de creatinine : pas d'evidence d'IRA -> stage 0
# (convention identique a celle utilisee pour les chiffres deja publies dans le manuscrit)
n_missing_creat = int(df2['AKI_stage'].isna().sum())
if n_missing_creat:
    print(f"NOTE: {n_missing_creat} patients sans aucune creatinine mesuree -> AKI_stage/AKI_KDIGO mis a 0 par defaut")
    df2['AKI_stage'] = df2['AKI_stage'].fillna(0)
    df2['AKI_KDIGO'] = df2['AKI_KDIGO'].fillna(0).astype(int)

print("=== Jeu de donnees final ===")
print("Patients :", len(df2))
print("AKI_KDIGO :", int(df2['AKI_KDIGO'].sum()), f"({100*df2['AKI_KDIGO'].mean():.1f}%)")
print("AKI_stage distribution :")
print(df2['AKI_stage'].value_counts(dropna=False).sort_index())
print("MAKE28 :", int(df2['MAKE28'].sum()))
print("MAKE90 :", int(df2['MAKE90'].sum()))

df2.to_excel(f"{OUT}/dffinal_clean.xlsx", index=False)
episodes_long.to_excel(f"{OUT}/episodes_long.xlsx", index=False)
hypo_episode_full.to_excel(f"{OUT}/hypo_episode_full.xlsx", index=False)
hyper_episode_full.to_excel(f"{OUT}/hyper_episode_full.xlsx", index=False)

print()
print("Exporte dans", OUT, ":")
print(" - dffinal_clean.xlsx (jeu de donnees final, 1 ligne/patient)")
print(" - episodes_long.xlsx")
print(" - hypo_episode_full.xlsx")
print(" - hyper_episode_full.xlsx")


NOTE: 22 patients sans aucune creatinine mesuree -> AKI_stage/AKI_KDIGO mis a 0 par defaut
=== Jeu de donnees final ===
Patients : 1758
AKI_KDIGO : 404 (23.0%)
AKI_stage distribution :
AKI_stage
0.0    1354
1.0     179
2.0      73
3.0     152
Name: count, dtype: int64
MAKE28 : 200
MAKE90 : 207



Exporte dans outputs :
 - dffinal_clean.xlsx (jeu de donnees final, 1 ligne/patient)
 - episodes_long.xlsx
 - hypo_episode_full.xlsx
 - hyper_episode_full.xlsx


## 25. Export des natrémies unitaires avec leur source (R1.3 — concordance ISE directe / indirecte)

Une ligne par mesure de `df_na_final` (fenêtre veille d'admission → sortie), avec :
- `source` : `lab` (laboratoire central, ISE indirecte, cobas) ou `bga` (gaz du sang, ISE directe) ;
- les valeurs brutes (`NA  :  Sodium`, `NAGZA` artériel, `NAGZV` veineux), `NA_raw_best` (après exclusion glycémie ≥ 10 mmol/L / TG ≥ 2 g/L) et `NA_corr_best` (valeur utilisée dans toutes les analyses) ;
- l'albumine et les protéines totales du même jour (moyenne journalière, laboratoire).

⚠️ Les gaz du sang de l'extraction sont datés au **jour** (pas d'heure) : l'appariement labo/gaz se fait donc par patient-jour dans le notebook 03.

In [26]:
# R1.3 : export des natremies unitaires + source + albumine/proteines du jour
nat_long = df_na_final[['IPP2', 'Date de Prélèvement', 'NA  :  Sodium', 'NAGZA  :  Sodium', 'NAGZV  :  Sodium',
                        'NA_raw_best', 'NA_corr_best', 'glu_daily_max_mmolL', 'tg_daily_max_gL']].copy()
nat_long['source'] = np.where(nat_long['NA  :  Sodium'].notna(), 'lab',
                     np.where(nat_long[['NAGZA  :  Sodium', 'NAGZV  :  Sodium']].notna().any(axis=1), 'bga', 'none'))
nat_long['day'] = pd.to_datetime(nat_long['Date de Prélèvement']).dt.normalize()

prot = dfglimps_all[['IPP2', 'Date -heure de prvt-', 'ALB  :  Albumine', 'PROTP  :  Protéines']].copy()
prot['day'] = pd.to_datetime(prot['Date -heure de prvt-']).dt.normalize()
for c in ['ALB  :  Albumine', 'PROTP  :  Protéines']:
    prot[c] = pd.to_numeric(prot[c].astype(str).str.replace('<', '', regex=False).str.replace(',', '.', regex=False), errors='coerce')
prot_daily = (prot.groupby(['IPP2', 'day'], as_index=False)[['ALB  :  Albumine', 'PROTP  :  Protéines']].mean()
              .rename(columns={'ALB  :  Albumine': 'albumin_gL', 'PROTP  :  Protéines': 'protein_gL'}))
nat_long = nat_long.merge(prot_daily, on=['IPP2', 'day'], how='left')

print("Mesures (lignes) :", len(nat_long), "| NA_corr_best non manquante :", int(nat_long['NA_corr_best'].notna().sum()))
print(nat_long.loc[nat_long['NA_corr_best'].notna(), 'source'].value_counts())
nat_long.drop(columns='day').to_csv(f"{OUT}/natremia_measurements_long.csv", index=False)
print("Exporte :", f"{OUT}/natremia_measurements_long.csv")

Mesures (lignes) : 114654 | NA_corr_best non manquante : 50797
source
lab    39251
bga    11546
Name: count, dtype: int64


Exporte : outputs/natremia_measurements_long.csv
